In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import pandas as pd

FROZEN_DIR = "/content/drive/MyDrive/frozen_dataset"
FEATURE_DIR = "/content/drive/MyDrive/feature_engineering_v1"
BASELINE_METRICS_PATH = "/content/drive/MyDrive/day 15/baseline_v1/baseline_metrics.csv"
ENSEMBLE_METRICS_PATH = "/content/drive/MyDrive/day16/ensemble_v1/train_test_metrics.csv"
DAY17_METRICS_PATH = "/content/drive/MyDrive/day17/ann_v1/day17_metrics.csv"
FULL_PATH = f"/content/drive/MyDrive/frozen_dataset/dataset_v1_full.csv"

TRAIN_PATH = f"/content/drive/MyDrive/frozen_dataset/dataset_v1_train.csv"
TEST_PATH = f"/content/drive/MyDrive/frozen_dataset/dataset_v1_test.csv"
ENGINEERED_PATH = f"/content/drive/MyDrive/feature_engineering_v1/engineered_dataset.csv"

TARGET_COLUMN = "soc_percent"
RAW_FEATURES = ["voltage_v", "current_a", "cell_temp_c", "power_w", "ambient_temp_c"]
CHARGE_TATRGET = "remaining_charge_time_s"
CHARGE_FEATURES = ["voltage_v", "current_a", "cell_temp_c", "ambient_temp_c"]


SOC_OVERRIDE = None

for p in [TRAIN_PATH, TEST_PATH]:
    assert os.path.exists(p), f"Couldn't find {p} — check the Day 12 copy step."

df_train_raw = pd.read_csv(TRAIN_PATH)
df_test_raw = pd.read_csv(TEST_PATH)          # touched ONLY in the final evaluation cell
print(f"Frozen split loaded — train: {df_train_raw.shape}, test (untouched until the end): {df_test_raw.shape}")

HAVE_ENGINEERED = os.path.exists(ENGINEERED_PATH)
if HAVE_ENGINEERED:
    df_eng_full = pd.read_csv(ENGINEERED_PATH)
    key_cols = ["file_id", "timestamp"]
    eng_cols = [c for c in df_eng_full.columns if c not in key_cols + [TARGET_COLUMN]]
    train_ids = df_train_raw["file_id"].unique()
    test_ids = df_test_raw["file_id"].unique()
    df_eng_train = df_eng_full[df_eng_full["file_id"].isin(train_ids)].copy()
    df_eng_test = df_eng_full[df_eng_full["file_id"].isin(test_ids)].copy()
    print(f"Engineered features ({len(eng_cols)}): {eng_cols}")

Frozen split loaded — train: (76792, 7), test (untouched until the end): (27768, 7)
Engineered features (2): ['voltage_roll_std5', 'current_roll_std5']


In [4]:
prior = []
if os.path.exists(BASELINE_METRICS_PATH):
    prior.append(pd.read_csv(BASELINE_METRICS_PATH, index_col=0)[["MAE", "RMSE", "R2"]])
if os.path.exists(ENSEMBLE_METRICS_PATH):
    ens = pd.read_csv(ENSEMBLE_METRICS_PATH, index_col=0)
    ens = ens[ens.index.str.endswith("(test)")].copy()
    ens.index = ens.index.str.replace(" (test)", "", regex=False)
    prior.append(ens[["MAE", "RMSE", "R2"]])
if os.path.exists(DAY17_METRICS_PATH):
    prior.append(pd.read_csv(DAY17_METRICS_PATH, index_col=0)[["MAE", "RMSE", "R2"]])

assert prior, ("No prior metrics files found — copy Day 15/16/17 outputs to Drive "
               "first, or hardcode SOC_OVERRIDE and skip auto-detection.")
all_prior = pd.concat(prior)
print("===== EVERY METHOD SCORED SO FAR =====")
print(all_prior.sort_values("MAE").to_string())

===== EVERY METHOD SCORED SO FAR =====
                                      MAE          RMSE        R2
label                                                            
RandomForest — raw               3.284465     10.474381  0.848121
RandomForest — raw               3.284465     10.474381  0.848121
XGBoost — raw                    3.300615     10.082012  0.859287
XGBoost — raw                    3.300615     10.082012  0.859287
Linear — raw                     3.472758      4.585786  0.970888
Linear — raw                     3.472758      4.585786  0.970888
XGBoost — engineered            22.598706     27.527405 -0.049465
XGBoost — engineered            22.598706     27.527405 -0.049465
RandomForest — engineered       22.942698     28.916572 -0.158060
RandomForest — engineered       22.942698     28.916572 -0.158060
Mean baseline                   23.041381     27.508849 -0.047575
Mean baseline                   23.041381     27.508849 -0.047575
Linear — engineered             23.19

In [7]:
TUNABLE_KEYWORDS = ["RandomForest", "GradientBoosting", "XGBoost", "MLP"]
soc_rows = all_prior[~all_prior.index.str.contains("charging", case=False)]
tunable_soc = soc_rows[soc_rows.index.to_series().apply(
    lambda label: any(k in label for k in TUNABLE_KEYWORDS)
)]
assert not tunable_soc.empty, "No tunable (tree/ANN) SOC method found in prior metrics."

if SOC_OVERRIDE:
    soc_shortlist_label = SOC_OVERRIDE
    print(f"\nSOC_OVERRIDE set — tuning '{soc_shortlist_label}' regardless of metrics.")
else:
    soc_shortlist_label = tunable_soc["MAE"].idxmin()
    print(f"\nAuto-detected SOC shortlist (lowest test MAE among tunable methods): "
          f"'{soc_shortlist_label}' (MAE={tunable_soc.loc[soc_shortlist_label, 'MAE'].iloc[0]:.3f})")

uses_engineered = "engineered" in soc_shortlist_label or "MLP" in soc_shortlist_label
if "RandomForest" in soc_shortlist_label:
    SOC_FAMILY = "RandomForest"
elif "GradientBoosting" in soc_shortlist_label or "XGBoost" in soc_shortlist_label:
    SOC_FAMILY = "GradientBoosting"
elif "MLP" in soc_shortlist_label:
    SOC_FAMILY = "MLP"
else:
    raise ValueError(f"Couldn't map '{soc_shortlist_label}' to a tunable family — set SOC_OVERRIDE.")

print(f"Family to tune: {SOC_FAMILY}  |  Feature set: {'engineered' if uses_engineered else 'raw'}")
print("\nSecond shortlisted candidate (fixed, per Day 17/18): Random Forest for "
      "remaining_charge_time_s — the only method justified for that target.")


Auto-detected SOC shortlist (lowest test MAE among tunable methods): 'RandomForest — raw' (MAE=3.284)
Family to tune: RandomForest  |  Feature set: raw

Second shortlisted candidate (fixed, per Day 17/18): Random Forest for remaining_charge_time_s — the only method justified for that target.


In [8]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

soc_features = eng_cols if uses_engineered else RAW_FEATURES
soc_train_df = df_eng_train if uses_engineered else df_train_raw

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
tr_idx, val_idx = next(gss_val.split(soc_train_df, groups=soc_train_df["file_id"]))
df_fit = soc_train_df.iloc[tr_idx]
df_val = soc_train_df.iloc[val_idx]

X_fit, y_fit = df_fit[soc_features], df_fit[TARGET_COLUMN]
X_val, y_val = df_val[soc_features], df_val[TARGET_COLUMN]
print(f"Validation carve — fit: {X_fit.shape}, val: {X_val.shape} "
      f"(grouped by file_id, same technique as Day 17's ANN validation split)")

def eval_mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)

Validation carve — fit: (62120, 5), val: (14672, 5) (grouped by file_id, same technique as Day 17's ANN validation split)


In [9]:
from sklearn.model_selection import ParameterGrid

experiment_log = []

if SOC_FAMILY == "RandomForest":
    from sklearn.ensemble import RandomForestRegressor
    print("===== SEARCH STRATEGY: Random Forest =====")
    print("Grid (not random) — small enough to run exhaustively and log every "
          "combination. Axes chosen because they control the three main ways "
          "an RF over/under-fits: how many trees vote (n_estimators), how deep "
          "each tree can split (max_depth), and how much a leaf must generalize "
          "before it's allowed to exist (min_samples_leaf).")
    param_grid = {
        "n_estimators": [150, 300, 450],
        "max_depth": [8, 12, 16],
        "min_samples_leaf": [1, 2, 4],
    }
    print(f"Grid size: {len(list(ParameterGrid(param_grid)))} configurations\n")

    for params in ParameterGrid(param_grid):
        model = RandomForestRegressor(random_state=42, n_jobs=-1, **params)
        model.fit(X_fit, y_fit)
        val_mae = eval_mae(y_val, model.predict(X_val))
        row = {**params, "val_MAE": val_mae}
        experiment_log.append(row)
        print(f"  {params} -> val_MAE={val_mae:.4f}")

elif SOC_FAMILY == "GradientBoosting":
    try:
        from xgboost import XGBRegressor
        BUILD = lambda **p: XGBRegressor(random_state=42, n_jobs=-1, subsample=0.8, colsample_bytree=0.8, **p)
        GB_NAME = "XGBoost"
    except ImportError:
        from sklearn.ensemble import GradientBoostingRegressor
        BUILD = lambda **p: GradientBoostingRegressor(random_state=42, subsample=0.8, **p)
        GB_NAME = "GradientBoosting"
    print(f"===== SEARCH STRATEGY: {GB_NAME} =====")
    print("Grid chosen around the three levers that most affect boosting's "
          "bias/variance trade-off: how many boosting rounds (n_estimators), "
          "how much each round can shift predictions (learning_rate), and how "
          "complex each individual tree is allowed to be (max_depth).")
    param_grid = {
        "n_estimators": [150, 300],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.03, 0.05, 0.1],
    }
    print(f"Grid size: {len(list(ParameterGrid(param_grid)))} configurations\n")

    for params in ParameterGrid(param_grid):
        model = BUILD(**params)
        model.fit(X_fit, y_fit)
        val_mae = eval_mae(y_val, model.predict(X_val))
        row = {**params, "val_MAE": val_mae}
        experiment_log.append(row)
        print(f"  {params} -> val_MAE={val_mae:.4f}")

elif SOC_FAMILY == "MLP":
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    from sklearn.preprocessing import StandardScaler

    scaler = StandardScaler().fit(X_fit)
    X_fit_s = scaler.transform(X_fit)
    X_val_s = scaler.transform(X_val)

    print("===== SEARCH STRATEGY: MLP =====")
    print("Grid over the levers that most affect an MLP's capacity vs. "
          "overfitting on this dataset: hidden-layer sizes (architecture), "
          "dropout rate, and learning rate. Epochs are capped with early "
          "stopping in every run so the grid compares configurations fairly, "
          "not just who trained longest.")
    param_grid = {
        "architecture": [(32, 16), (64, 32), (128, 64, 32)],
        "dropout": [0.1, 0.2],
        "lr": [1e-3, 5e-4],
    }
    print(f"Grid size: {len(list(ParameterGrid(param_grid)))} configurations\n")

    def build_mlp(architecture, dropout, lr, n_features):
        m = keras.Sequential([keras.Input(shape=(n_features,))])
        for units in architecture:
            m.add(layers.Dense(units, activation="relu"))
            m.add(layers.Dropout(dropout))
        m.add(layers.Dense(1, activation="linear"))
        m.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss="mse", metrics=["mae"])
        return m

    for params in ParameterGrid(param_grid):
        tf.random.set_seed(42)
        model = build_mlp(n_features=X_fit_s.shape[1], **params)
        es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
        model.fit(X_fit_s, y_fit, validation_data=(X_val_s, y_val),
                  epochs=80, batch_size=256, callbacks=[es], verbose=0)
        val_pred = model.predict(X_val_s, verbose=0).flatten()
        val_mae = eval_mae(y_val, val_pred)
        row = {**params, "architecture": str(params["architecture"]), "val_MAE": val_mae}
        experiment_log.append(row)
        print(f"  {params} -> val_MAE={val_mae:.4f}")

experiment_log_df = pd.DataFrame(experiment_log).sort_values("val_MAE").reset_index(drop=True)
print("\n===== EXPERIMENT LOG (sorted best-first) =====")
print(experiment_log_df.to_string())

best_params = experiment_log_df.iloc[0].drop("val_MAE").to_dict()
if SOC_FAMILY == "MLP":
    best_params["architecture"] = eval(best_params["architecture"])  # back to tuple
else:
    best_params = {k: (int(v) if float(v).is_integer() else v) for k, v in best_params.items()}
print(f"\nBest validated configuration: {best_params} "
      f"(val_MAE={experiment_log_df.iloc[0]['val_MAE']:.4f})")


===== SEARCH STRATEGY: Random Forest =====
Grid (not random) — small enough to run exhaustively and log every combination. Axes chosen because they control the three main ways an RF over/under-fits: how many trees vote (n_estimators), how deep each tree can split (max_depth), and how much a leaf must generalize before it's allowed to exist (min_samples_leaf).
Grid size: 27 configurations

  {'max_depth': 8, 'min_samples_leaf': 1, 'n_estimators': 150} -> val_MAE=1.2580
  {'max_depth': 8, 'min_samples_leaf': 1, 'n_estimators': 300} -> val_MAE=1.2493
  {'max_depth': 8, 'min_samples_leaf': 1, 'n_estimators': 450} -> val_MAE=1.2474
  {'max_depth': 8, 'min_samples_leaf': 2, 'n_estimators': 150} -> val_MAE=1.2553
  {'max_depth': 8, 'min_samples_leaf': 2, 'n_estimators': 300} -> val_MAE=1.2489
  {'max_depth': 8, 'min_samples_leaf': 2, 'n_estimators': 450} -> val_MAE=1.2488
  {'max_depth': 8, 'min_samples_leaf': 4, 'n_estimators': 150} -> val_MAE=1.2491
  {'max_depth': 8, 'min_samples_leaf': 4,

In [12]:
import numpy as np
os.makedirs("tuning_v1", exist_ok=True)

X_train_full = soc_train_df[soc_features]
y_train_full = soc_train_df[TARGET_COLUMN]
soc_test_df = df_eng_test if uses_engineered else df_test_raw
X_test_final = soc_test_df[soc_features]
y_test_final = soc_test_df[TARGET_COLUMN]

if SOC_FAMILY == "RandomForest":
    from sklearn.ensemble import RandomForestRegressor
    final_model = RandomForestRegressor(random_state=42, n_jobs=-1, **best_params)
    final_model.fit(X_train_full, y_train_full)
    y_pred_final = final_model.predict(X_test_final)
    import joblib
    joblib.dump(final_model, "tuning_v1/soc_tuned_model.joblib")

elif SOC_FAMILY == "GradientBoosting":
    final_model = BUILD(**best_params)
    final_model.fit(X_train_full, y_train_full)
    y_pred_final = final_model.predict(X_test_final)
    import joblib
    joblib.dump(final_model, "tuning_v1/soc_tuned_model.joblib")

elif SOC_FAMILY == "MLP":
    scaler_full = StandardScaler().fit(X_train_full)
    X_train_full_s = scaler_full.transform(X_train_full)
    X_test_final_s = scaler_full.transform(X_test_final)
    tf.random.set_seed(42)
    final_model = build_mlp(n_features=X_train_full_s.shape[1], **best_params)
    es = keras.callbacks.EarlyStopping(monitor="loss", patience=8, restore_best_weights=True)
    final_model.fit(X_train_full_s, y_train_full, epochs=80, batch_size=256, callbacks=[es], verbose=0)
    y_pred_final = final_model.predict(X_test_final_s, verbose=0).flatten()
    final_model.save("tuning_v1/soc_tuned_model.keras")
    import joblib
    joblib.dump(scaler_full, "tuning_v1/soc_tuned_scaler.joblib")

soc_final_mae = mean_absolute_error(y_test_final, y_pred_final)
soc_final_rmse = np.sqrt(mean_squared_error(y_test_final, y_pred_final))
soc_final_r2 = r2_score(y_test_final, y_pred_final)
print("\n===== FINAL TEST RESULT (test set touched for the first time, once) =====")
print(f"Tuned {SOC_FAMILY} — MAE={soc_final_mae:.3f}  RMSE={soc_final_rmse:.3f}  R2={soc_final_r2:.4f}")
print(f"Compare to untuned '{soc_shortlist_label}': "
      f"MAE={all_prior.loc[soc_shortlist_label, 'MAE'].iloc[0]:.3f}  "
      f"({'improved' if soc_final_mae < all_prior.loc[soc_shortlist_label, 'MAE'].iloc[0] else 'no improvement — keep the untuned config'})")

experiment_log_df.to_csv("tuning_v1/soc_experiment_log.csv", index=False)


===== FINAL TEST RESULT (test set touched for the first time, once) =====
Tuned RandomForest — MAE=2.899  RMSE=8.379  R2=0.9028
Compare to untuned 'RandomForest — raw': MAE=3.284  (improved)


In [29]:
print("\n\n===== SHORTLISTED CANDIDATE 2: Remaining Charging Time (Random Forest) =====")

assert os.path.exists(FULL_PATH), f"Couldn't find {FULL_PATH} for charging-time features."
df_full = pd.read_csv(FULL_PATH)
df_full["prog_time_td"] = pd.to_timedelta(df_full["prog_time"], errors="coerce")
df_full = df_full.sort_values(["file_id", "prog_time_td"]).reset_index(drop=True)
df_full["status_change"] = ((df_full["status"] != df_full["status"].shift(1)) |
                             (df_full["file_id"] != df_full["file_id"].shift(1)))
df_full["segment_id"] = df_full["status_change"].cumsum()
df_charge = df_full[df_full["status"] == "CHA"].copy()
seg_end = df_charge.groupby("segment_id")["prog_time_td"].transform("max")
df_charge[CHARGE_TATRGET] = (seg_end - df_charge["prog_time_td"]).dt.total_seconds()


charge_train_full = df_charge[df_charge["file_id"].isin(df_train_raw["file_id"].unique())].copy()
charge_test_full = df_charge[df_charge["file_id"].isin(df_test_raw["file_id"].unique())].copy()

# Drop rows where the target variable is NaN
charge_train_full.dropna(subset=[CHARGE_TATRGET], inplace=True)
charge_test_full.dropna(subset=[CHARGE_TATRGET], inplace=True)

if len(charge_train_full) < 20 or charge_train_full["file_id"].nunique() < 2:
    print("⚠️  Too few charging rows/files to run a controlled search reliably — skipping.")
else:
    gss_c = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
    c_tr_idx, c_val_idx = next(gss_c.split(charge_train_full, groups=charge_train_full["file_id"]))
    c_fit = charge_train_full.iloc[c_tr_idx]
    c_val = charge_train_full.iloc[c_val_idx]

    print("===== SEARCH STRATEGY: Random Forest (charging time) ====")
    print("Smaller grid than SOC's — deliberately, because the charging dataset "
          "is much smaller and a large grid risks picking a config that overfits "
          "the validation split itself, not just the training data.")
    charge_grid = {"n_estimators": [100, 200, 300], "max_depth": [6, 10, 14]}
    print(f"Grid size: {len(list(ParameterGrid(charge_grid)))} configurations\n")

    from sklearn.ensemble import RandomForestRegressor
    charge_log = []
    for params in ParameterGrid(charge_grid):
        model = RandomForestRegressor(random_state=42, n_jobs=-1, **params)
        model.fit(c_fit[CHARGE_FEATURES], c_fit[CHARGE_TATRGET])
        val_mae = eval_mae(c_val[CHARGE_TATRGET], model.predict(c_val[CHARGE_FEATURES]))
        charge_log.append({**params, "val_MAE": val_mae})
        print(f"  {params} -> val_MAE={val_mae:.4f}")

    charge_log_df = pd.DataFrame(charge_log).sort_values("val_MAE").reset_index(drop=True)
    charge_log_df.to_csv("tuning_v1/charging_time_experiment_log.csv", index=False)
    print("\n===== CHARGING-TIME EXPERIMENT LOG (best-first) ====")
    print(charge_log_df.to_string())

    charge_best_params = {k: int(v) for k, v in charge_log_df.iloc[0].drop("val_MAE").to_dict().items()}
    print(f"\nBest validated configuration: {charge_best_params} "
          f"(val_MAE={charge_log_df.iloc[0]['val_MAE']:.4f})")

    final_charge_model = RandomForestRegressor(random_state=42, n_jobs=-1, **charge_best_params)
    final_charge_model.fit(charge_train_full[CHARGE_FEATURES], charge_train_full[CHARGE_TATRGET])
    y_pred_charge = final_charge_model.predict(charge_test_full[CHARGE_FEATURES])

    charge_mae = mean_absolute_error(charge_test_full[CHARGE_TATRGET], y_pred_charge)
    charge_rmse = np.sqrt(mean_squared_error(charge_test_full[CHARGE_TATRGET], y_pred_charge))
    charge_r2 = r2_score(charge_test_full[CHARGE_TATRGET], y_pred_charge)
    print(f"\nFINAL TEST RESULT — tuned RF (charging time): "
          f"MAE={charge_mae:.3f}s  RMSE={charge_rmse:.3f}s  R2={charge_r2:.4f}")

    import joblib
    joblib.dump(final_charge_model, "tuning_v1/charging_time_tuned_model.joblib")



===== SHORTLISTED CANDIDATE 2: Remaining Charging Time (Random Forest) =====
===== SEARCH STRATEGY: Random Forest (charging time) ====
Smaller grid than SOC's — deliberately, because the charging dataset is much smaller and a large grid risks picking a config that overfits the validation split itself, not just the training data.
Grid size: 9 configurations

  {'max_depth': 6, 'n_estimators': 100} -> val_MAE=31215.5115
  {'max_depth': 6, 'n_estimators': 200} -> val_MAE=31217.7342
  {'max_depth': 6, 'n_estimators': 300} -> val_MAE=31243.1975
  {'max_depth': 10, 'n_estimators': 100} -> val_MAE=31213.7602
  {'max_depth': 10, 'n_estimators': 200} -> val_MAE=31215.8461
  {'max_depth': 10, 'n_estimators': 300} -> val_MAE=31241.2495
  {'max_depth': 14, 'n_estimators': 100} -> val_MAE=31214.1378
  {'max_depth': 14, 'n_estimators': 200} -> val_MAE=31216.2993
  {'max_depth': 14, 'n_estimators': 300} -> val_MAE=31241.5140

===== CHARGING-TIME EXPERIMENT LOG (best-first) ====
   max_depth  n_esti

In [30]:
import datetime
with open("tuning_v1/summary.txt", "w") as f:
    f.write("Day 19 — Controlled Hyperparameter / Parameter Experiments\n")
    f.write(f"Generated: {datetime.datetime.now()}\n\n")
    f.write(f"Shortlisted SOC candidate: {soc_shortlist_label} (family: {SOC_FAMILY}, "
            f"features: {'engineered' if uses_engineered else 'raw'})\n")
    f.write("Selection rule: validation split carved from TRAINING data only "
            "(GroupShuffleSplit, 15%, grouped by file_id) — the Day 12 test set "
            "was not touched until the single final evaluation below.\n\n")
    f.write(f"Best validated config: {best_params}\n")
    f.write(f"Final test MAE={soc_final_mae:.3f}, RMSE={soc_final_rmse:.3f}, R2={soc_final_r2:.4f}\n")
    f.write(f"Untuned baseline for comparison ('{soc_shortlist_label}'): "
            f"MAE={all_prior.loc[soc_shortlist_label, 'MAE'].iloc[0]:.3f}\n\n")
    f.write("Second shortlisted candidate: Random Forest for remaining_charge_time_s\n")
    if len(charge_train_full) >= 20 and charge_train_full["file_id"].nunique() >= 2:
        f.write(f"Best validated config: {charge_best_params}\n")
        f.write(f"Final test MAE={charge_mae:.3f}s, RMSE={charge_rmse:.3f}s, R2={charge_r2:.4f}\n")
    else:
        f.write("Skipped — insufficient charging data for a reliable controlled search.\n")

print("\n✅ Day 19 artifacts saved in 'tuning_v1/':")
for f in sorted(os.listdir("tuning_v1")):
    print(" -", f)


✅ Day 19 artifacts saved in 'tuning_v1/':
 - charging_time_experiment_log.csv
 - charging_time_tuned_model.joblib
 - soc_experiment_log.csv
 - soc_tuned_model.joblib
 - summary.txt


In [31]:
import shutil
shutil.copytree("tuning_v1", "/content/drive/MyDrive/Dataset_Li-ion/tuning_v1", dirs_exist_ok=True)

'/content/drive/MyDrive/Dataset_Li-ion/tuning_v1'